<a href="https://colab.research.google.com/github/geethat0126-bit/AI_5SEM/blob/main/week9_1BM24CS104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
class Expr: pass

class Var(Expr):
    def __init__(self, name): self.name = name
    def __str__(self): return self.name

class Func(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __str__(self): return f"{self.name}({', '.join(map(str, self.args))})" if self.args else self.name

class Pred(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __str__(self): return f"{self.name}({', '.join(map(str, self.args))})"

class Not(Expr):
    def __init__(self, op): self.op = op
    def __str__(self): return f"¬{self.op}" if isinstance(self.op, (Pred, Not)) else f"¬({self.op})"

class And(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ∧ {self.right})"

class Or(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ∨ {self.right})"

class Implies(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ⇒ {self.right})"

class Quant(Expr):
    def __init__(self, q, v, b): self.qtype, self.var, self.body = q, v, b
    def __str__(self): return f"{'∀' if self.qtype == 'FORALL' else '∃'}{self.var} [{self.body}]"

def tokenize(text):
    for k, v in {"∀":" forall ","∃":" exists ","⇒":" => ","->":" => ","∧":" & ","∨":" | ","¬":" ~ ","~":" ~ ","[":" ( ","]":" ) ","(":" ( ",")":" ) ",",":" , "}.items():
        text = text.replace(k, v)
    return text.split()

class Parser:
    def __init__(self, tokens): self.tokens, self.pos = tokens, 0
    def peek(self): return self.tokens[self.pos] if self.pos < len(self.tokens) else None
    def consume(self, exp=None):
        tok = self.peek()
        if not tok or (exp and tok != exp): raise ValueError(f"Expected {exp}, got {tok}")
        self.pos += 1
        return tok
    def parse(self):
        e = self.parse_implies()
        if self.peek(): raise ValueError(f"Unexpected token: {self.peek()}")
        return e
    def parse_implies(self):
        l = self.parse_or()
        while self.peek() == "=>": self.consume("=>"); l = Implies(l, self.parse_or())
        return l
    def parse_or(self):
        l = self.parse_and()
        while self.peek() == "|": self.consume("|"); l = Or(l, self.parse_and())
        return l
    def parse_and(self):
        l = self.parse_unary()
        while self.peek() == "&": self.consume("&"); l = And(l, self.parse_unary())
        return l
    def parse_unary(self):
        t = self.peek()
        if t == "~": self.consume("~"); return Not(self.parse_unary())
        if t in ("forall", "exists"): return Quant("FORALL" if self.consume() == "forall" else "EXISTS", self.consume(), self.parse_unary())
        if t == "(": self.consume("("); e = self.parse_implies(); self.consume(")"); return e
        name = self.consume(); self.consume("("); args = []
        if self.peek() != ")":
            args.append(self.parse_term())
            while self.peek() == ",": self.consume(","); args.append(self.parse_term())
        self.consume(")")
        return Pred(name, args)
    def parse_term(self):
        name = self.consume()
        if self.peek() == "(":
            self.consume("("); args = [self.parse_term()]
            while self.peek() == ",": self.consume(","); args.append(self.parse_term())
            self.consume(")")
            return Func(name, args)
        return Var(name) if name.islower() else Func(name, [])

def elim_imp(e):
    if isinstance(e, Implies): return Or(Not(elim_imp(e.left)), elim_imp(e.right))
    if isinstance(e, Not): return Not(elim_imp(e.op))
    if isinstance(e, And): return And(elim_imp(e.left), elim_imp(e.right))
    if isinstance(e, Or): return Or(elim_imp(e.left), elim_imp(e.right))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, elim_imp(e.body))
    return e

def move_not(e):
    if isinstance(e, Not):
        i = e.op
        if isinstance(i, Not): return move_not(i.op)
        if isinstance(i, And): return Or(move_not(Not(i.left)), move_not(Not(i.right)))
        if isinstance(i, Or): return And(move_not(Not(i.left)), move_not(Not(i.right)))
        if isinstance(i, Quant): return Quant("EXISTS" if i.qtype == "FORALL" else "FORALL", i.var, move_not(Not(i.body)))
        return Not(move_not(i))
    if isinstance(e, And): return And(move_not(e.left), move_not(e.right))
    if isinstance(e, Or): return Or(move_not(e.left), move_not(e.right))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, move_not(e.body))
    return e

def std_vars(e, used=None, m=None):
    used, m = used or set(), m or {}
    if isinstance(e, Var): return Var(m.get(e.name, e.name))
    if isinstance(e, Func): return Func(e.name, [std_vars(a, used, m) for a in e.args])
    if isinstance(e, Pred): return Pred(e.name, [std_vars(a, used, m) for a in e.args])
    if isinstance(e, Not): return Not(std_vars(e.op, used, m))
    if isinstance(e, And): return And(std_vars(e.left, used, m), std_vars(e.right, used, m))
    if isinstance(e, Or): return Or(std_vars(e.left, used, m), std_vars(e.right, used, m))
    if isinstance(e, Quant):
        nv = e.var
        while nv in used: nv = chr(ord(nv) + 1) if nv < 'z' else nv + "_1"
        used.add(nv); nm = dict(m); nm[e.var] = nv
        return Quant(e.qtype, nv, std_vars(e.body, used, nm))
    return e

sk_count = 0
def subst(e, v, rep):
    if isinstance(e, Var): return rep if e.name == v else e
    if isinstance(e, Func): return Func(e.name, [subst(a, v, rep) for a in e.args])
    if isinstance(e, Pred): return Pred(e.name, [subst(a, v, rep) for a in e.args])
    if isinstance(e, Not): return Not(subst(e.op, v, rep))
    if isinstance(e, And): return And(subst(e.left, v, rep), subst(e.right, v, rep))
    if isinstance(e, Or): return Or(subst(e.left, v, rep), subst(e.right, v, rep))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, subst(e.body, v, rep))
    return e

def skolemize(e, uvars=None):
    global sk_count
    uvars = uvars or []
    if isinstance(e, Quant):
        if e.qtype == "FORALL": return Quant("FORALL", e.var, skolemize(e.body, uvars + [e.var]))
        sk = chr(ord('F') + sk_count); sk_count += 1
        return skolemize(subst(e.body, e.var, Func(sk, [Var(v) for v in uvars])), uvars)
    if isinstance(e, Not): return Not(skolemize(e.op, uvars))
    if isinstance(e, And): return And(skolemize(e.left, uvars), skolemize(e.right, uvars))
    if isinstance(e, Or): return Or(skolemize(e.left, uvars), skolemize(e.right, uvars))
    return e

def drop_univ(e):
    if isinstance(e, Quant): return drop_univ(e.body)
    if isinstance(e, Not): return Not(drop_univ(e.op))
    if isinstance(e, And): return And(drop_univ(e.left), drop_univ(e.right))
    if isinstance(e, Or): return Or(drop_univ(e.left), drop_univ(e.right))
    return e

def dist_or(e):
    if isinstance(e, And): return And(dist_or(e.left), dist_or(e.right))
    if isinstance(e, Or):
        a, b = dist_or(e.left), dist_or(e.right)
        if isinstance(a, And): return And(dist_or(Or(a.left, b)), dist_or(Or(a.right, b)))
        if isinstance(b, And): return And(dist_or(Or(a, b.left)), dist_or(Or(a, b.right)))
        return Or(a, b)
    return e

def get_clauses(e):
    return get_clauses(e.left) + get_clauses(e.right) if isinstance(e, And) else [str(e)]

if __name__ == "__main__":
    print("Symbols: forall (∀), exists (∃), => (⇒), & (∧), | (∨), ~ (¬)")
    s = input("Enter FOL Formula: ").strip()
    if s:
        e = Parser(tokenize(s)).parse()
        print("\nOriginal:", e)
        e = elim_imp(e); print("1. Eliminate implications:", e)
        e = move_not(e); print("2. Move ¬ inwards:", e)
        e = std_vars(e); print("3. Standardize variables:", e)
        e = skolemize(e); print("4. Skolemization:", e)
        e = drop_univ(e); print("5. Drop universal quantifiers:", e)
        e = dist_or(e); print("6. Distribute ∨ over ∧ (CNF):", e)
        print("\n7. Final CNF Clauses:")
        for i, c in enumerate(get_clauses(e), 1): print(f"   Clause {i}: {c}")


Symbols: forall (∀), exists (∃), => (⇒), & (∧), | (∨), ~ (¬)
Enter FOL Formula: ∀x [∀y Animal(y) ⇒ Loves(x, y)] ⇒ [∃y Loves(y, x)]

Original: (∀x [(∀y [Animal(y)] ⇒ Loves(x, y))] ⇒ ∃y [Loves(y, x)])
1. Eliminate implications: (¬(∀x [(¬(∀y [Animal(y)]) ∨ Loves(x, y))]) ∨ ∃y [Loves(y, x)])
2. Move ¬ inwards: (∃x [(∀y [Animal(y)] ∧ ¬Loves(x, y))] ∨ ∃y [Loves(y, x)])
3. Standardize variables: (∃x [(∀y [Animal(y)] ∧ ¬Loves(x, y))] ∨ ∃y [Loves(y, x)])
4. Skolemization: ((∀y [Animal(y)] ∧ ¬Loves(F, y)) ∨ Loves(G, x))
5. Drop universal quantifiers: ((Animal(y) ∧ ¬Loves(F, y)) ∨ Loves(G, x))
6. Distribute ∨ over ∧ (CNF): ((Animal(y) ∨ Loves(G, x)) ∧ (¬Loves(F, y) ∨ Loves(G, x)))

7. Final CNF Clauses:
   Clause 1: (Animal(y) ∨ Loves(G, x))
   Clause 2: (¬Loves(F, y) ∨ Loves(G, x))


In [4]:
class Expr: pass

class Var(Expr):
    def __init__(self, n): self.name = n
    def __str__(self): return self.name

class Func(Expr):
    def __init__(self, n, a): self.name, self.args = n, a
    def __str__(self): return f"{self.name}({','.join(map(str, self.args))})" if self.args else self.name

class Pred(Expr):
    def __init__(self, n, a): self.name, self.args = n, a
    def __str__(self): return f"{self.name}({','.join(map(str, self.args))})"

class Not(Expr):
    def __init__(self, o): self.op = o
    def __str__(self): return f"¬{self.op}" if isinstance(self.op, (Pred, Not)) else f"¬({self.op})"

class And(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ∧ {self.right})"

class Or(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ∨ {self.right})"

class Implies(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} → {self.right})"

class Quant(Expr):
    def __init__(self, q, v, b): self.qtype, self.var, self.body = q, v, b
    def __str__(self): return f"({'∀' if self.qtype == 'FORALL' else '∃'}{self.var}){self.body}"

def tokenize(text):
    for k, v in {"∀":" forall ","∃":" exists ","→":" -> ","⇒":" -> ","^":" & ","∧":" & ","∨":" | ","¬":" ~ ","~":" ~ ","[":" ( ","]":" ) ","(":" ( ",")":" ) ",",":" , "}.items():
        text = text.replace(k, v)
    return text.split()

class Parser:
    def __init__(self, tokens): self.tokens, self.pos = tokens, 0
    def peek(self): return self.tokens[self.pos] if self.pos < len(self.tokens) else None
    def consume(self, exp=None):
        t = self.peek()
        if not t or (exp and t != exp): raise ValueError(f"Expected {exp}, got {t}")
        self.pos += 1
        return t
    def parse(self):
        e = self.parse_imp()
        if self.peek(): raise ValueError(f"Unexpected: {self.peek()}")
        return e
    def parse_imp(self):
        l = self.parse_or()
        while self.peek() == "->": self.consume("->"); l = Implies(l, self.parse_or())
        return l
    def parse_or(self):
        l = self.parse_and()
        while self.peek() == "|": self.consume("|"); l = Or(l, self.parse_and())
        return l
    def parse_and(self):
        l = self.parse_unary()
        while self.peek() == "&": self.consume("&"); l = And(l, self.parse_unary())
        return l
    def parse_unary(self):
        t = self.peek()
        if t == "~": self.consume("~"); return Not(self.parse_unary())
        if t == "(":
            if self.pos + 1 < len(self.tokens) and self.tokens[self.pos + 1] in ("forall", "exists"):
                self.consume("(")
                q = "FORALL" if self.consume() == "forall" else "EXISTS"
                v = self.consume()
                self.consume(")")
                return Quant(q, v, self.parse_unary())
            self.consume("("); e = self.parse_imp(); self.consume(")"); return e
        if t in ("forall", "exists"):
            return Quant("FORALL" if self.consume() == "forall" else "EXISTS", self.consume(), self.parse_unary())
        name = self.consume(); self.consume("("); args = []
        if self.peek() != ")":
            args.append(self.parse_term())
            while self.peek() == ",": self.consume(","); args.append(self.parse_term())
        self.consume(")")
        return Pred(name, args)
    def parse_term(self):
        name = self.consume()
        if self.peek() == "(":
            self.consume("("); args = [self.parse_term()]
            while self.peek() == ",": self.consume(","); args.append(self.parse_term())
            self.consume(")")
            return Func(name, args)
        return Var(name) if name.islower() else Func(name, [])

def elim_imp(e):
    if isinstance(e, Implies): return Or(Not(elim_imp(e.left)), elim_imp(e.right))
    if isinstance(e, Not): return Not(elim_imp(e.op))
    if isinstance(e, And): return And(elim_imp(e.left), elim_imp(e.right))
    if isinstance(e, Or): return Or(elim_imp(e.left), elim_imp(e.right))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, elim_imp(e.body))
    return e

def move_not(e):
    if isinstance(e, Not):
        i = e.op
        if isinstance(i, Not): return move_not(i.op)
        if isinstance(i, And): return Or(move_not(Not(i.left)), move_not(Not(i.right)))
        if isinstance(i, Or): return And(move_not(Not(i.left)), move_not(Not(i.right)))
        if isinstance(i, Quant): return Quant("EXISTS" if i.qtype == "FORALL" else "FORALL", i.var, move_not(Not(i.body)))
        return Not(move_not(i))
    if isinstance(e, And): return And(move_not(e.left), move_not(e.right))
    if isinstance(e, Or): return Or(move_not(e.left), move_not(e.right))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, move_not(e.body))
    return e

def std_vars(e, used=None, m=None):
    used, m = used if used is not None else set(), m if m is not None else {}
    if isinstance(e, Var): return Var(m.get(e.name, e.name))
    if isinstance(e, Func): return Func(e.name, [std_vars(a, used, m) for a in e.args])
    if isinstance(e, Pred): return Pred(e.name, [std_vars(a, used, m) for a in e.args])
    if isinstance(e, Not): return Not(std_vars(e.op, used, m))
    if isinstance(e, And): return And(std_vars(e.left, used, m), std_vars(e.right, used, m))
    if isinstance(e, Or): return Or(std_vars(e.left, used, m), std_vars(e.right, used, m))
    if isinstance(e, Quant):
        nv = e.var
        while nv in used: nv = 'z' if nv == 'x' else ('w' if nv == 'y' else chr(ord(nv) + 1))
        used.add(nv); nm = dict(m); nm[e.var] = nv
        return Quant(e.qtype, nv, std_vars(e.body, used, nm))
    return e

sk_count = 0
def subst(e, v, rep):
    if isinstance(e, Var): return rep if e.name == v else e
    if isinstance(e, Func): return Func(e.name, [subst(a, v, rep) for a in e.args])
    if isinstance(e, Pred): return Pred(e.name, [subst(a, v, rep) for a in e.args])
    if isinstance(e, Not): return Not(subst(e.op, v, rep))
    if isinstance(e, And): return And(subst(e.left, v, rep), subst(e.right, v, rep))
    if isinstance(e, Or): return Or(subst(e.left, v, rep), subst(e.right, v, rep))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, subst(e.body, v, rep))
    return e

def skolemize(e, uvars=None):
    global sk_count
    uvars = uvars if uvars is not None else []
    if isinstance(e, Quant):
        if e.qtype == "FORALL": return Quant("FORALL", e.var, skolemize(e.body, uvars + [e.var]))
        sk = chr(ord('c') + sk_count) if not uvars else chr(ord('F') + sk_count)
        sk_count += 1
        return skolemize(subst(e.body, e.var, Func(sk, [Var(v) for v in uvars])), uvars)
    if isinstance(e, Not): return Not(skolemize(e.op, uvars))
    if isinstance(e, And): return And(skolemize(e.left, uvars), skolemize(e.right, uvars))
    if isinstance(e, Or): return Or(skolemize(e.left, uvars), skolemize(e.right, uvars))
    return e

def drop_univ(e):
    if isinstance(e, Quant): return drop_univ(e.body)
    if isinstance(e, Not): return Not(drop_univ(e.op))
    if isinstance(e, And): return And(drop_univ(e.left), drop_univ(e.right))
    if isinstance(e, Or): return Or(drop_univ(e.left), drop_univ(e.right))
    return e

def dist_or(e):
    if isinstance(e, And): return And(dist_or(e.left), dist_or(e.right))
    if isinstance(e, Or):
        a, b = dist_or(e.left), dist_or(e.right)
        if isinstance(a, And): return And(dist_or(Or(a.left, b)), dist_or(Or(a.right, b)))
        if isinstance(b, And): return And(dist_or(Or(a, b.left)), dist_or(Or(a, b.right)))
        return Or(a, b)
    return e

def flatten_and(e):
    return flatten_and(e.left) + flatten_and(e.right) if isinstance(e, And) else [str(e)]

if __name__ == "__main__":
    s = input("Enter FOL formula: ").strip()
    e = Parser(tokenize(s)).parse()
    e = elim_imp(e)
    print("1. Eliminate Implications:", e)
    e = move_not(e)
    print("2. Move Negations Inward:", e)
    e = std_vars(e)
    print("3. Standardize Variables:", e)
    e = skolemize(e)
    print("4. Skolemization:", e)
    e = drop_univ(e)
    print("5. Drop Universal Quantifiers:", e)
    e = dist_or(e)
    print("6. Conjunctive Normal Form:", " ∧ ".join(flatten_and(e)))


Enter FOL formula: ~(forall x)(exists y)[~p(x) -> ~q(x,y)] & (exists x)(forall y)q(x,y)
1. Eliminate Implications: (¬((∀x)(∃y)(¬¬p(x) ∨ ¬q(x,y))) ∧ (∃x)(∀y)q(x,y))
2. Move Negations Inward: ((∃x)(∀y)(¬p(x) ∧ q(x,y)) ∧ (∃x)(∀y)q(x,y))
3. Standardize Variables: ((∃x)(∀y)(¬p(x) ∧ q(x,y)) ∧ (∃z)(∀w)q(z,w))
4. Skolemization: ((∀y)(¬p(c) ∧ q(c,y)) ∧ (∀w)q(d,w))
5. Drop Universal Quantifiers: ((¬p(c) ∧ q(c,y)) ∧ q(d,w))
6. Conjunctive Normal Form: ¬p(c) ∧ q(c,y) ∧ q(d,w)


In [13]:
class Var:
    def __init__(self, n): self.name = n
    def __str__(self): return self.name
    def __eq__(self, o): return isinstance(o, Var) and self.name == o.name
    def __hash__(self): return hash(('V', self.name))

class Func:
    def __init__(self, n, a=None): self.name, self.args = n, a or []
    def __str__(self): return f"{self.name}({', '.join(map(str, self.args))})" if self.args else self.name
    def __eq__(self, o): return isinstance(o, Func) and self.name == o.name and self.args == o.args
    def __hash__(self): return hash(('F', self.name, tuple(self.args)))

class Lit:
    def __init__(self, neg, pred, args): self.neg, self.pred, self.args = neg, pred, args
    def __str__(self): return f"{'¬' if self.neg else ''}{self.pred}({', '.join(map(str, self.args))})"
    def __eq__(self, o): return isinstance(o, Lit) and self.neg == o.neg and self.pred == o.pred and self.args == o.args
    def __hash__(self): return hash((self.neg, self.pred, tuple(self.args)))

def tokenize(s):
    for k, v in {"∀":" forall ","∃":" exists ","→":" -> ","⇒":" -> ","^":" & ","∧":" & ","∨":" | ","¬":" ~ ","~":" ~ ","[":" ( ","]":" ) ","(":" ( ",")":" ) ",",":" , ",":":" "}.items():
        s = s.replace(k, v)
    return s.split()

class Parser:
    def __init__(self, tok): self.tok, self.pos = tok, 0
    def peek(self): return self.tok[self.pos] if self.pos < len(self.tok) else None
    def eat(self, e=None):
        t = self.peek()
        if not t or (e and t != e): raise ValueError(f"Expected {e}, got {t}")
        self.pos += 1; return t
    def parse(self):
        r = self.imp()
        if self.peek(): raise ValueError(f"Unexpected {self.peek()}")
        return r
    def imp(self):
        l = self.or_e()
        while self.peek() == "->": self.eat("->"); l = ('IMP', l, self.or_e())
        return l
    def or_e(self):
        l = self.and_e()
        while self.peek() == "|": self.eat("|"); l = ('OR', l, self.and_e())
        return l
    def and_e(self):
        l = self.un()
        while self.peek() == "&": self.eat("&"); l = ('AND', l, self.un())
        return l
    def un(self):
        t = self.peek()
        if t == "~": self.eat("~"); return ('NOT', self.un())
        if t in ("forall", "exists"): return ("FORALL" if self.eat() == "forall" else "EXISTS", self.eat(), self.un())
        if t == "(": self.eat("("); e = self.imp(); self.eat(")"); return e
        n = self.eat(); self.eat("("); a = []
        if self.peek() != ")":
            a.append(self.term())
            while self.peek() == ",": self.eat(","); a.append(self.term())
        self.eat(")")
        return ('PRED', n, a)
    def term(self):
        n = self.eat()
        if self.peek() == "(":
            self.eat("("); a = [self.term()]
            while self.peek() == ",": self.eat(","); a.append(self.term())
            self.eat(")"); return Func(n, a)
        return Var(n) if n.islower() else Func(n, [])

def elim_imp(e):
    op = e[0]
    if op == 'IMP': return ('OR', ('NOT', elim_imp(e[1])), elim_imp(e[2]))
    if op == 'NOT': return ('NOT', elim_imp(e[1]))
    if op in ('AND', 'OR'): return (op, elim_imp(e[1]), elim_imp(e[2]))
    if op in ('FORALL', 'EXISTS'): return (op, e[1], elim_imp(e[2]))
    return e

def nnf(e):
    op = e[0]
    if op == 'NOT':
        i = e[1]
        if i[0] == 'NOT': return nnf(i[1])
        if i[0] == 'AND': return ('OR', nnf(('NOT', i[1])), nnf(('NOT', i[2])))
        if i[0] == 'OR': return ('AND', nnf(('NOT', i[1])), nnf(('NOT', i[2])))
        if i[0] in ('FORALL', 'EXISTS'): return ('EXISTS' if i[0] == 'FORALL' else 'FORALL', i[1], nnf(('NOT', i[2])))
        return ('NOT', i)
    if op in ('AND', 'OR'): return (op, nnf(e[1]), nnf(e[2]))
    if op in ('FORALL', 'EXISTS'): return (op, e[1], nnf(e[2]))
    return e

sk_id = 0
def subst_t(t, v, r):
    return (r if t.name == v else t) if isinstance(t, Var) else Func(t.name, [subst_t(a, v, r) for a in t.args])

def subst_e(e, v, r):
    op = e[0]
    if op == 'PRED': return ('PRED', e[1], [subst_t(a, v, r) for a in e[2]])
    if op == 'NOT': return ('NOT', subst_e(e[1], v, r))
    if op in ('AND', 'OR'): return (op, subst_e(e[1], v, r), subst_e(e[2], v, r))
    if op in ('FORALL', 'EXISTS'): return (op, e[1], subst_e(e[2], v, r))
    return e

def skolem(e, u=None):
    global sk_id
    u, op = u or [], e[0]
    if op == 'FORALL': return skolem(e[2], u + [e[1]])
    if op == 'EXISTS':
        sk = f"SK{sk_id}"; sk_id += 1
        return skolem(subst_e(e[2], e[1], Func(sk, [Var(x) for x in u])), u)
    if op in ('AND', 'OR'): return (op, skolem(e[1], u), skolem(e[2], u))
    return e

def dist(e):
    if e[0] == 'AND': return ('AND', dist(e[1]), dist(e[2]))
    if e[0] == 'OR':
        a, b = dist(e[1]), dist(e[2])
        if a[0] == 'AND': return ('AND', dist(('OR', a[1], b)), dist(('OR', a[2], b)))
        if b[0] == 'AND': return ('AND', dist(('OR', a, b[1])), dist(('OR', a, b[2])))
        return ('OR', a, b)
    return e

def to_clauses(e):
    if e[0] == 'AND': return to_clauses(e[1]) + to_clauses(e[2])
    lits = lambda x: lits(x[1]) + lits(x[2]) if x[0] == 'OR' else ([Lit(True, x[1][1], x[1][2])] if x[0] == 'NOT' else [Lit(False, x[1], x[2])])
    return [lits(e)]

def fol_to_cnf(s):
    return to_clauses(dist(skolem(nnf(elim_imp(Parser(tokenize(s)).parse())))))

def apply_sub_t(t, s):
    return (apply_sub_t(s[t.name], s) if t.name in s else t) if isinstance(t, Var) else Func(t.name, [apply_sub_t(a, s) for a in t.args])

def apply_sub_c(c, s):
    return [Lit(l.neg, l.pred, [apply_sub_t(a, s) for a in l.args]) for l in c]

def occurs(v, t, s):
    t = apply_sub_t(t, s)
    return t.name == v if isinstance(t, Var) else any(occurs(v, a, s) for a in t.args)

def unify(x, y, s=None):
    s = {} if s is None else s
    if isinstance(x, list):
        if len(x) != len(y): return None
        for a, b in zip(x, y):
            s = unify(a, b, s)
            if s is None: return None
        return s
    x, y = apply_sub_t(x, s), apply_sub_t(y, s)
    if x == y: return s
    if isinstance(x, Var):
        if occurs(x.name, y, s): return None
        ns = dict(s); ns[x.name] = y; return ns
    if isinstance(y, Var):
        if occurs(y.name, x, s): return None
        ns = dict(s); ns[y.name] = x; return ns
    if isinstance(x, Func) and isinstance(y, Func) and x.name == y.name:
        return unify(x.args, y.args, s)
    return None

var_uid = 0
def standardize_clause(c):
    global var_uid
    var_uid += 1; m = {}
    ren = lambda t: (m.setdefault(t.name, Var(f"{t.name}_{var_uid}")) if isinstance(t, Var) else Func(t.name, [ren(a) for a in t.args]))
    return [Lit(l.neg, l.pred, [ren(a) for a in l.args]) for l in c]

fmt = lambda c: "{  }  Hence proved." if not c else " ∨ ".join(map(str, c))

def resolve_pair(c1, c2):
    c1_s, c2_s, res = standardize_clause(c1), standardize_clause(c2), []
    for i, l1 in enumerate(c1_s):
        for j, l2 in enumerate(c2_s):
            if l1.pred == l2.pred and l1.neg != l2.neg:
                sub = unify(l1.args, l2.args)
                if sub is not None:
                    nc = apply_sub_c([l for k, l in enumerate(c1_s) if k != i] + [l for k, l in enumerate(c2_s) if k != j], sub)
                    uc = []
                    for lit in nc:
                        if lit not in uc: uc.append(lit)
                    csub = {k.split('_')[0]: str(v) for k, v in sub.items() if not str(v).startswith(k.split('_')[0])}
                    res.append((uc, csub))
    return res

def print_tree(chain):
    print("\n--- PROOF BY RESOLUTION TREE ---\n")
    for st, (lc, rc, sub, res) in enumerate(chain):
        ss = "{" + ", ".join(f"{v}/{k}" for k, v in sub.items()) + "}" if sub else ""
        print(f"  {fmt(lc) if st == 0 else '':<28} {fmt(rc)}")
        print(f"      \\                      /\n       \\                    /    {ss}\n        \\                  /\n         ▼                ▼\n      {fmt(res)}")

def prove(kb, neg_goal):
    clauses = [(c, None) for c in kb] + [(ng, None) for ng in neg_goal]
    q, vis = list(range(len(kb), len(clauses))), {tuple(c) for c, _ in clauses}
    while q:
        i1 = q.pop(0)
        for i2 in range(len(clauses)):
            if i1 == i2: continue
            for res, sub in resolve_pair(clauses[i1][0], clauses[i2][0]):
                if not res:
                    chain = []
                    def tr(a, b, s, r):
                        if clauses[a][1] is not None: tr(*clauses[a][1])
                        chain.append((clauses[a][0], clauses[b][0], s, r))
                    tr(i1, i2, sub, res); print_tree(chain); return
                if tuple(res) not in vis:
                    vis.add(tuple(res)); clauses.append((res, (i1, i2, sub, res))); q.append(len(clauses) - 1)
    print("Could not prove.")

n = int(input("Enter number of premises: "))
kb = [c for i in range(n) for c in fol_to_cnf(input(f"Premise {i+1}: "))]
ng = fol_to_cnf(f"~({input('Enter Goal: ')})")
for i, c in enumerate(kb, 1): print(f"{i}. {fmt(c)}")
for c in ng: print(f"Negated Goal: {fmt(c)}")
prove(kb, ng)


Enter number of premises: 7
Premise 1: forall x food(x) -> likes(John, x)
Premise 2: food(Apple) & food(vegetables)
Premise 3: forall x forall y (eats(x, y) & ~killed(x)) -> food(y)
Premise 4: eats(Anil, Peanuts) & alive(Anil)
Premise 5: forall x eats(Anil, x) -> eats(Harry, x)
Premise 6: forall x ~killed(x) -> alive(x)
Premise 7: forall x alive(x) -> ~killed(x)
Enter Goal: likes(John, Peanuts)
1. ¬food(SK0) ∨ likes(John, x)
2. food(Apple)
3. food(vegetables)
4. ¬eats(SK1, SK2) ∨ killed(SK1) ∨ food(y)
5. eats(Anil, Peanuts)
6. alive(Anil)
7. ¬eats(Anil, SK3) ∨ eats(Harry, x)
8. killed(SK4) ∨ alive(x)
9. ¬alive(SK5) ∨ ¬killed(x)
Negated Goal: ¬likes(John, Peanuts)

--- PROOF BY RESOLUTION TREE ---

  ¬likes(John, Peanuts)        ¬food(SK0) ∨ likes(John, x)
      \                      /
       \                    /    {Peanuts/x}
        \                  /
         ▼                ▼
      ¬food(SK0)
                               food(vegetables)
      \                      /
     